In [ ]:
%matplotlib inline 

In [ ]:
# Imports
import os
import pathlib
import urllib.request

# Site name for Salvus Flow. Uses env var if available, otherwise falls back to local site name.
SALVUS_FLOW_SITE_NAME = os.environ.get("SALVUS_FLOW_SITE_NAME", "salome_remote_2")
PROJECT_DIR = "simulation_wavefield_custom_stf_weighted" 

# Conservative default to reduce license-seat demand during unstable license-server periods.
RANKS_PER_JOB = 4


def check_license_server_reachable(url="https://l.mondaic.com/licensing_server", timeout=10):
    """Fail fast if licensing endpoint is unreachable to avoid long hanging jobs."""
    try:
        with urllib.request.urlopen(url, timeout=timeout):
            return True
    except Exception as exc:
        raise RuntimeError(
            f"Licensing server not reachable ({url}). Retry later or check network/VPN. Original error: {exc}"
        ) from exc


# Add code to keep .gitignore updated to ignore salvus files
gitignore_path = pathlib.Path("..") / ".gitignore"
with open(gitignore_path, "r+") as f:
    contents = f.read()
    if PROJECT_DIR not in contents:
        f.write(f"\n{PROJECT_DIR}/\n")


import numpy as np
import salvus.namespace as sn
import salvus.flow.simple_config as sc
import xarray as xr
import salvus.namespace as sn
from salvus.project.tools.processing import block_processing
from salvus.toolbox.helpers.wavefield_output import (
    WavefieldOutput,
    wavefield_output_to_xarray,
)
import matplotlib.pyplot as plt
from matplotlib import animation
import obspy

# For wavefield output code
from salvus.mesh.unstructured_mesh_utils import read_model_from_h5
from salvus.toolbox.helpers import wavefield_output
import salvus.flow.api

#for plotting of wiggles, traces 
from scipy import signal

# For animation plot
from IPython.display import HTML
from matplotlib.collections import PolyCollection
import threading
import matplotlib
matplotlib.use("Agg")
from scipy.interpolate import griddata

In [ ]:
# Setup of the model domain as a box (same as research module)
domain_2d = sn.domain.dim2.BoxDomain(x0=0, x1=400, y0=0, y1=3)
p = sn.Project.from_domain(path=PROJECT_DIR, domain=domain_2d, load_if_exists=True)

In [ ]:
# Source setup with weights

# SImulation constants
f0 = 20.0 # Center frequency of Ricker(Hz)
sampling_rate = 10000.0 # Sampling rate (Hz)
dt = 1.0 / sampling_rate # Time step in seconds
t_max = 3.0 # Total simulation duration (seconds)

# Fixed delay step between consecutive sources (seconds)
# calculating this from x positions
step = 50 # distance between the sources in m
x_positions = np.arange(30.0, 270.0, step)
target_vprop = 90.0 # target velocity, sub-rayleigh or supershear
delay_between_sources = step / target_vprop
print(f"Calculated time step between sources based on target velocity: {delay_between_sources:.4f} s") 
y_src = 2.625             

# Define a local symmetric time window for a single base wavelet
wavelet_half_width = 0.08  
t_local = np.arange(-wavelet_half_width, wavelet_half_width, dt)

# Padding for the first source so there isn't noise in the grid
t_start_padding = 1.5 * wavelet_half_width

# Create the master timeline that matches  full simulation duration
t_sim = np.arange(0, t_max, dt)

# Generate symmetric base Ricker wavelet centered at t_local = 0
wavelet_base = (1.0 - 2.0 * (np.pi * f0 * t_local) ** 2) * np.exp(-((np.pi * f0 * t_local) ** 2)) # formula for ricker from seg wiki: f(t)=(1-2\pi ^{2}f_{_{M}}^{2}t^{2})e^{-\pi ^{2}f_{_{M}}^{2}t^{2}}


srcs = []

for i, x_src in enumerate(x_positions):
    # Calculate relative delay based on the source index
    relative_delay = t_start_padding + (i * delay_between_sources)
    
    # Initialize an empty master simulation timeline for this specific source
    wavelet_delayed = np.zeros_like(t_sim)
    
    # Map the local centered wavelet samples to their delayed index window
    center_sample = int(relative_delay * sampling_rate)
    half_samples = len(wavelet_base) // 2
    
    start_idx = center_sample - half_samples
    end_idx = center_sample + half_samples
    
    # insert the wavelet within the boundaries of the master simulation time
    if end_idx > 0 and start_idx < len(t_sim):
        sim_slice = slice(max(0, start_idx), min(len(t_sim), end_idx))
        wav_slice = slice(max(0, -start_idx), min(len(wavelet_base), len(t_sim) - start_idx))
        wavelet_delayed[sim_slice] = wavelet_base[wav_slice]

    # Isoltaed scaling for first source only
    if i == 0:
        
        amplitude_scale = 0.5  
    else:
        
        amplitude_scale = 1.0

    scaled_myy = -1.54e+01 * amplitude_scale

    # Initialize the custom STF forcing start_time_in_seconds to 0
    stf = sc.stf.Custom.from_array(
        array=wavelet_delayed,
        sampling_rate_in_hertz=sampling_rate,
        start_time_in_seconds=0.0
    )
    
    # plotting every 100 sources to check the delay visually in the STF
    plotting_steps = np.arange(0, len(x_positions), 100)
    if i in plotting_steps:
        print(f"Plotting source index {i} at x = {x_src} m (calculated delay = {relative_delay:.4f} s):")
        stf.plot()
        fig = plt.gcf()
        from IPython.display import display
        display(fig)
        plt.close(fig)

    # Wrap inside Salvus point source configuration
    src = sc.source.cartesian.MomentTensorPoint2D(
        x=x_src,  
        y=y_src, # Taken from setup 
        mxx=0.0,
        myy=scaled_myy,
        mxy=0.0,
        source_time_function=stf
    )

    srcs.append(src)

print(f"Successfully generated {len(srcs)} sequentially delayed sources.")